# Exercise: Flood Analysis in Opole (2024) using Sentinel-1 SAR Data

In this exercise, you will compare radar backscatter from Sentinel-1 before and after the 2024 flood in Opole, Poland.

**Objective:**
- Load Sentinel-1 VV data before and after the flood
- Convert radar intensity to decibels (dB)
- Calculate the difference between the two dates
- Identify areas with a significant decrease in VV (possible flood zones)

## Step 1: Import Earth Engine and geemap

In [3]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='rsia-lab')
Map = geemap.Map(center=[50.675, 17.931], zoom=12)
region = ee.Geometry.Point([17.931, 50.675])

## Step 2: Load Sentinel-1 VV images (before and after flood)

In [5]:
# TODO: Load Sentinel-1 collection for before and after flood
# Use filterBounds, filterDate, and select 'VV' band
# Load Sentinel-1 collection for before and after flood
# Example dates: before = '2024-05-15' to '2024-05-25', after = '2024-05-30' to '2024-06-05'
before = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filterBounds(region) \
    .filterDate('2024-05-15', '2024-05-25') \
    .filter(ee.Filter.eq('instrumentMode', 'IW')) \
    .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING')) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
    .select('VV')

after = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filterBounds(region) \
    .filterDate('2024-05-30', '2024-06-05') \
    .filter(ee.Filter.eq('instrumentMode', 'IW')) \
    .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING')) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
    .select('VV')


## Step 3: Convert to dB scale

In [7]:
# TODO: Use .log10().multiply(10) to convert before/after images to dB

before_db = before.mean().log10().multiply(10)
after_db = after.mean().log10().multiply(10)

## Step 4: Calculate difference and threshold potential flood areas

In [9]:
# TODO: Subtract after_dB from before_dB
# Create flood mask: pixels where difference > 3 dB

difference = before_db.subtract(after_db)
flood_mask = difference.gt(3)



## Step 5: Visualize the results

In [12]:
# TODO: Use Map.addLayer() to show before, after, difference, and flood mask
Map.addLayer(before_db, {'min': -30, 'max': 0}, 'Before Flood (dB)')
Map.addLayer(after_db, {'min': -30, 'max': 0}, 'After Flood (dB)')
Map.addLayer(difference, {'min': -10, 'max': 10}, 'Difference (dB)')
Map.addLayer(flood_mask.updateMask(flood_mask), {'palette': ['blue']}, 'Flood Mask')

Map

Map(bottom=88270.0, center=[50.557069943985425, 17.946166992187504], controls=(WidgetControl(options=['positio…